In [1]:
import requests
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import csv
import time

# URL of the sitemap
sitemap_url = 'https://hebbarskitchen.com/post-sitemap.xml'

# Step 1: Fetch sitemap and filter recipe URLs
response = requests.get(sitemap_url)
root = ET.fromstring(response.content)
recipe_keywords = ['recipe', 'food']

recipe_urls = []
for url in root.findall('{http://www.sitemaps.org/schemas/sitemap/0.9}url'):
    loc = url.find('{http://www.sitemaps.org/schemas/sitemap/0.9}loc').text
    if any(keyword in loc for keyword in recipe_keywords):
        recipe_urls.append(loc)

print(f"Found {len(recipe_urls)} recipe URLs")

# Step 2: Scrape details and write to CSV
with open('hebbars_indian_recipes.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Dish Name', 'Ingredients', 'Instructions'])

    for index, url in enumerate(recipe_urls):
        try:
            res = requests.get(url, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Dish name
            title_tag = soup.find('h1', class_='entry-title')
            if not title_tag:
                continue
            dish_name = title_tag.text.strip()

            # Ingredients
            ingredients = []
            for ing in soup.find_all('li', class_='wprm-recipe-ingredient'):
                qty = ing.find('span', class_='wprm-recipe-ingredient-amount')
                qty = qty.text.strip() if qty else "N/A"

                name = ing.find('span', class_='wprm-recipe-ingredient-name')
                name = name.text.strip() if name else "N/A"

                ingredients.append(f"{qty} {name}")
            ingredients_text = '\n'.join(ingredients)

            # Instructions
            instructions = []
            for step in soup.find_all('div', class_='wprm-recipe-instruction-text'):
                if step.text.strip():
                    instructions.append(step.text.strip())
            instructions_text = '\n'.join(instructions)

            writer.writerow([dish_name, ingredients_text, instructions_text])
            print(f"[{index+1}/{len(recipe_urls)}] Scraped: {dish_name}")

            time.sleep(1)  # polite crawling

        except Exception as e:
            print(f"Failed at {url} -> {e}")

Found 926 recipe URLs
[1/926] Scraped: paneer recipes
[2/926] Scraped: why skipping breakfast is bad & how to deal it with healthy food intakes
[3/926] Scraped: ghevar recipe | how to make crispy & porous ghewar at home
[4/926] Scraped: misal pav recipe | how to make maharashtrian misal pav recipe
[5/926] Scraped: seviyan kheer recipe | semiya payasam | semiya kheer or vermicelli kheer
[6/926] Scraped: brinjal fry recipe | brinjal rava fry | baingan rava fry | fried eggplant
[7/926] Scraped: rajma pulao recipe | kidney beans pulao | rajma beans pulao
[8/926] Scraped: eggless pancake recipe | pancakes without eggs | eggless pancakes
[9/926] Scraped: sukha puri recipe | stuffed sukha poori chaat | sukha masala puri
[10/926] Scraped: honey cake recipe | how to make eggless bakery style honey cake
[11/926] Scraped: cabbage chutney recipe | cabbage pachadi recipe | muttaikose chutney
[12/926] Scraped: strawberry panna cotta recipe | strawberry panna cotta without gelatin
[13/926] Scraped: a

In [3]:
import pandas as pd
import re

# Load the CSV
df = pd.read_csv("hebbars_indian_recipes.csv")

# Drop rows where dish name or ingredients are missing
df.dropna(subset=['Dish Name', 'Ingredients'], inplace=True)

# Clean Dish Names
df['Dish Name'] = df['Dish Name'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()).title())

# Clean Ingredients
def clean_ingredients(text):
    lines = str(text).split('\n')
    cleaned = []
    for line in lines:
        line = re.sub(r'\s+', ' ', line.strip())  # remove extra spaces
        line = re.sub(r'[^a-zA-Z0-9.,()\- ]', '', line)  # remove special chars except common ones
        if line:
            cleaned.append(line)
    return '\n'.join(cleaned)

df['Ingredients'] = df['Ingredients'].apply(clean_ingredients)

# Optional: Clean Instructions if the column exists
if 'Instructions' in df.columns:
    def clean_instructions(text):
        text = str(text)
        text = re.sub(r'\n+', '. ', text)  # convert line breaks to sentence breaks
        text = re.sub(r'Step \d+:?|^\d+\.', '', text, flags=re.MULTILINE)  # remove step numbers
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    df['Instructions'] = df['Instructions'].apply(clean_instructions)

# Remove duplicate dishes
df.drop_duplicates(subset='Dish Name', keep='first', inplace=True)

# Save the cleaned dataset
df.to_csv("indian_recipes_cleaned.csv", index=False)

print("Cleaning complete. Saved as 'indian_recipes_cleaned.csv'")

Cleaning complete. Saved as 'indian_recipes_cleaned.csv'


In [5]:
import pandas as pd
import re

# Load the CSV
df = pd.read_csv("hebbars_indian_recipes.csv")

# Drop rows with missing data
df.dropna(subset=['Dish Name', 'Ingredients'], inplace=True)

# Function to check if a string is mostly English
def is_mostly_english(text, threshold=0.7):
    english_chars = re.findall(r'[a-zA-Z\s]', text)
    return len(english_chars) / len(text) >= threshold

# Filter out dish names that aren't mostly English
df = df[df['Dish Name'].apply(lambda x: is_mostly_english(str(x)))]

# Clean Dish Names
df['Dish Name'] = df['Dish Name'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()).title())

# Clean Ingredients
def clean_ingredients(text):
    lines = str(text).split('\n')
    cleaned = []
    for line in lines:
        line = re.sub(r'\s+', ' ', line.strip())
        line = re.sub(r'[^a-zA-Z0-9.,()\- ]', '', line)
        if line:
            cleaned.append(line)
    return '\n'.join(cleaned)

df['Ingredients'] = df['Ingredients'].apply(clean_ingredients)

# Optional: Clean Instructions if the column exists
if 'Instructions' in df.columns:
    def clean_instructions(text):
        text = str(text)
        text = re.sub(r'\n+', '. ', text)
        text = re.sub(r'(Step\s?\d+:?|^\d+\.)', '', text, flags=re.MULTILINE)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    df['Instructions'] = df['Instructions'].apply(clean_instructions)

# Drop duplicates based on Dish Name
df.drop_duplicates(subset='Dish Name', keep='first', inplace=True)

# Save cleaned CSV
df.to_csv("indian_recipes_cleaned.csv", index=False)

print("✅ Cleaned dataset saved as 'indian_recipes_cleaned.csv'")


✅ Cleaned dataset saved as 'indian_recipes_cleaned.csv'


In [6]:
import pandas as pd

# Load cleaned CSV
df = pd.read_csv("indian_recipes_cleaned.csv")

# Convert to list of dictionaries
data = df.to_dict(orient='records')

# Save as JSON
import json
with open("indian_recipes_cleaned.json", "w", encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ JSON saved as 'indian_recipes_cleaned.json'")


✅ JSON saved as 'indian_recipes_cleaned.json'
